## Transformer Architecture for Time-Series Classification

### 1. What is a Transformer?
A Transformer is a deep learning architecture built entirely on the **attention mechanism**,
discarding recurrence (RNN/LSTM) and convolution entirely. Originally proposed by
Vaswani et al. (2017) in *"Attention Is All You Need"*, it processes an entire sequence
in parallel, making it significantly faster to train than sequential models.

Its core equation is the **Scaled Dot-Product Attention**:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Where:
- $Q$ = Query matrix — *"what am I looking for?"*
- $K$ = Key matrix — *"what do I contain?"*
- $V$ = Value matrix — *"what do I return if matched?"*
- $d_k$ = key dimension (scaling factor to prevent vanishing gradients in softmax)

---

### 2. Multi-Head Attention (MHA)
Instead of computing attention once, MHA runs $h$ attention heads **in parallel**,
each projecting $Q, K, V$ into a different learned subspace:

$$\text{MHA}(Q,K,V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h)\,W^O$$

$$\text{where} \quad \text{head}_i = \text{Attention}(QW_i^Q,\ KW_i^K,\ VW_i^V)$$

| Head role | What it learns |
|-----------|---------------|
| Head 1 | Short-range local patterns |
| Head 2 | Long-range temporal dependencies |
| Head $h$ | Frequency-domain or amplitude features |

Each head specializes independently. The outputs are concatenated and projected
back via $W^O$, giving the model a **multi-perspective view** of the sequence.

---

### 3. Application to Time-Series Data
In time-series input $\mathbf{X} \in \mathbb{R}^{B \times L \times d}$ (batch × sequence
length × features), the Transformer treats each time step as a **token**. MHA then
computes pairwise relationships between every pair of time steps simultaneously:

$$A_{ij} = \frac{q_i \cdot k_j}{\sqrt{d_k}}$$

This produces an $L \times L$ attention map where $A_{ij}$ encodes how much
time step $i$ should attend to time step $j$ — capturing dependencies at
**any temporal distance** without the locality bias of CNNs or the forgetting
problem of RNNs.

**For fall detection specifically:**
- The pre-fall walking frames and the impact frame are compared directly,
  regardless of their distance in the sequence.
- The SVM spike at the impact frame receives disproportionately high attention
  weight, acting as a natural anchor across all heads.

---

### 4. Effect of Class Imbalance on Transformers

> ** Transformers are significantly affected by class imbalance,
> and in some ways more so than CNNs.**

The softmax in MHA normalizes attention scores **across all tokens**. When the
training set is dominated by the majority class (e.g., non-fall), the model
learns attention patterns biased toward that class distribution.

#### Why it hurts more than standard models:
The cross-entropy loss used in Transformer training:

$$\mathcal{L} = -\sum_{c} y_c \log(\hat{y}_c)$$

treats all samples equally. With severe imbalance, the gradient signal from
the minority class (falls) is **drowned out** by the majority class, causing
the model to collapse toward predicting non-fall for everything — achieving
high accuracy but catastrophic recall.

#### Mitigation strategies (in order of effectiveness):

| Strategy | How it helps |
|----------|-------------|
| **Weighted cross-entropy** | Scales minority class loss by $\frac{N}{n_{\text{minority}}}$ |
| **Focal loss** | Down-weights easy majority samples, focuses on hard minority cases |
| **SMOTE / oversampling** | Balances class distribution before training |
| **Threshold tuning** | Shifts decision boundary from 0.5 toward minority class |
| **Recall-weighted F1 monitoring** | Prevents accuracy from masking poor minority performance |

In the context of fall detection, **recall is the critical metric** — a missed
fall (false negative) is far more dangerous than a false alarm. Weighted loss
or focal loss is strongly recommended.

using attention mechanism


In [2]:
# ==========================================
# STANDARD & ML LIBRARIES
# ==========================================
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
import numpy as np
import math

In [3]:
# Import your custom MLOps Framework from the utilities folder
from utilities.dataset_router import DatasetRouter
from utilities.window_extractor import WindowExtractor
from utilities.experiment_trainer import ExperimentTrainer
from utilities.experiment_tester import ExperimentTester

model:conformer

## Why the CNN Precedes the Transformer

The 1D CNN is placed before the Transformer to solve two compounding problems:

### 1. Noise Suppression
Raw 50-frame accelerometer data is jagged and noisy. Feeding it directly into a
Transformer causes attention to waste capacity on irrelevant micro-vibrations.
The CNN sweeps across the sequence with a local kernel (~0.14s window), extracting
only physically meaningful events — wrist twist, free-fall, ground impact — before
the Transformer ever sees the data.

### 2. Quadratic Memory Reduction
Transformer attention scales as $\mathcal{O}(L^2)$. The CNN's pooling operation
halves the sequence length, cutting attention cost by 75%:

| Stage | Length | Attention Map |
|-------|--------|---------------|
| Raw input | 50 frames | $50 \times 50 = 2500$ |
| After CNN | 25 frames | $25 \times 25 = 625$ |

### Division of Labour
- **CNN** — local micro-physics: finds the spike, twist, and impact.
- **Transformer** — global macro-timeline: connects pre-fall walking → free-fall → impact
  into a single confident decision.

> The CNN cleans and compresses. The Transformer reasons and decides.

In [4]:
# ==========================================
# 0. THE MODEL ARCHITECTURE
# ==========================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=50):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :].to(x.device)

class ConvTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        # Tuned Hyperparameters from Optuna
        self.d_model = 128
        self.dropout_rate = 0.32
        
        # 10-Channel Input (9 Sensors + 1 SMV)
        self.conv1 = nn.Conv1d(10, 64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(64)
        self.pool = nn.MaxPool1d(2) 
        self.conv2 = nn.Conv1d(64, self.d_model, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(self.d_model)
        
        self.pos_encoder = PositionalEncoding(d_model=self.d_model, max_len=25)
        encoder_layers = nn.TransformerEncoderLayer(
            d_model=self.d_model, nhead=4, dim_feedforward=self.d_model*2, 
            dropout=self.dropout_rate, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=2)
        
        self.fc1 = nn.Linear(self.d_model, 64)
        self.dropout = nn.Dropout(self.dropout_rate)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = F.relu(self.bn2(self.conv2(x)))
        x = x.permute(0, 2, 1) 
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x)
        x = x.mean(dim=1) 
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x).squeeze(1)

In [5]:
# ==========================================
# 1. DATA ROUTING (Leak-Proof)
# ==========================================
# Assuming you have a Training folder and a separate Test folder
TRAIN_DIR = "./DataSet/Sample_Training"
TEST_DIR = "./DataSet/Sample_Test"

router = DatasetRouter(dataset_path=TRAIN_DIR, test_size=0.20, random_state=42)
train_files, val_files = router.create_splits()

test_router = DatasetRouter(dataset_path=TEST_DIR, test_size=0.01) # Hack to load 100% as test
test_files, _ = test_router.create_splits()


--- Initializing File Router on DataSet/Sample_Training ---
Found 2473 total CSV files. Tagging...


Routing Files: 100%|██████████| 2473/2473 [00:44<00:00, 55.30it/s] 



=== ROUTING SUCCESSFUL ===
Train Files: 1978 (Falls: 1489)
Val Files:   495 (Falls: 373)

--- Initializing File Router on DataSet/Sample_Test ---
Found 586 total CSV files. Tagging...


Routing Files: 100%|██████████| 586/586 [00:09<00:00, 59.32it/s] 


=== ROUTING SUCCESSFUL ===
Train Files: 580 (Falls: 435)
Val Files:   6 (Falls: 4)


In [6]:
# ==========================================
# 2. WINDOW EXTRACTION (The Engines)
# ==========================================
extractor = WindowExtractor(window_size=50, fall_threshold=0.4)

# Train gets the Dynamic Balancer Trick
X_train_raw, y_train, _ = extractor.extract_dynamic(train_files, normal_stride=25, fall_stride=5)

# Validation gets Standard real-world simulation
X_val_raw, y_val, _ = extractor.extract_standard(val_files, stride=25)

# Test gets the Strict 50% Overlap to prevent file-boundary contamination
X_test_raw, y_test, test_log = extractor.extract_strict_overlap(test_files, overlap_fraction=0.5)

Extracting (50% Overlap): 100%|██████████| 580/580 [00:07<00:00, 75.24it/s]


In [7]:
# ==========================================
# 3. FEATURE SCALING (Strictly Fitted to Train)
# ==========================================
print("\n--- Scaling Features ---")
scaler = StandardScaler()
n_tr, t_st, n_f = X_train_raw.shape
X_train_scaled = scaler.fit_transform(X_train_raw.reshape(-1, n_f)).reshape(n_tr, t_st, n_f)

n_v = X_val_raw.shape[0]
X_val_scaled = scaler.transform(X_val_raw.reshape(-1, n_f)).reshape(n_v, t_st, n_f)

n_ts = X_test_raw.shape[0]
X_test_scaled = scaler.transform(X_test_raw.reshape(-1, n_f)).reshape(n_ts, t_st, n_f)


--- Scaling Features ---


In [8]:
print(X_train_raw.shape)

(46781, 50, 10)


the one without hyperparam tuning

In [13]:
# ==========================================
# 4. TRAINING ENGINE
# ==========================================
EXP_NAME = "Exp_001__CNN+Trans"
model = ConvTransformer()
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0004) # Optuna's chosen LR

trainer = ExperimentTrainer(
    exp_name=EXP_NAME,
    description="10-Channel SMV. Dynamic Train, Standard Val",
    model=model,
    criterion=criterion,
    optimizer=optimizer
)

# Launch the loop
best_model, metrics = trainer.train(
    X_train_scaled, y_train, 
    X_val_scaled, y_val, 
    epochs=30, 
    batch_size=128
)


=== LAUNCHING EXPERIMENT: Exp_001__CNN+Trans ===
Device: cuda | Epochs: 30 | Batch: 128
Epoch [01/30] | Train Loss: 0.1180 | Val Loss: 0.0574 | Val F1: 0.8822 | Recall: 0.8592
Epoch [02/30] | Train Loss: 0.0635 | Val Loss: 0.0482 | Val F1: 0.9030 | Recall: 0.9150
Epoch [03/30] | Train Loss: 0.0561 | Val Loss: 0.0574 | Val F1: 0.8873 | Recall: 0.9551
Epoch [04/30] | Train Loss: 0.0494 | Val Loss: 0.0472 | Val F1: 0.9027 | Recall: 0.9066
Epoch [05/30] | Train Loss: 0.0467 | Val Loss: 0.0508 | Val F1: 0.9039 | Recall: 0.9417
Epoch [06/30] | Train Loss: 0.0435 | Val Loss: 0.0532 | Val F1: 0.8972 | Recall: 0.9211
Epoch [07/30] | Train Loss: 0.0413 | Val Loss: 0.0618 | Val F1: 0.8891 | Recall: 0.9393
Epoch [08/30] | Train Loss: 0.0378 | Val Loss: 0.0589 | Val F1: 0.8866 | Recall: 0.9442
Epoch [09/30] | Train Loss: 0.0376 | Val Loss: 0.0510 | Val F1: 0.9145 | Recall: 0.9284
Epoch [10/30] | Train Loss: 0.0343 | Val Loss: 0.0613 | Val F1: 0.9030 | Recall: 0.9260
Epoch [11/30] | Train Loss: 0.0

hyperparam tuning -Used Bayesian Search


In [8]:
import optuna
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import f1_score
import math

# ==========================================
# 1. MISSING DEPENDENCIES (Local to this cell)
# ==========================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=50):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :].to(x.device)

class FallDataset(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y
    def __len__(self): return len(self.y)
    def __getitem__(self, idx):
        # Transpose transforms (SeqLen, Channels) to (Channels, SeqLen) for 1D CNNs
        return torch.tensor(self.X[idx].T, dtype=torch.float32), torch.tensor(self.y[idx], dtype=torch.float32)


# ==========================================
# 2. THE TUNABLE ARCHITECTURE
# ==========================================
class TunableConvTransformer(nn.Module):
    def __init__(self, d_model, num_heads, num_layers, dropout_rate):
        super().__init__()
        
        # 10-Channel Input (9 Sensors + 1 SMV)
        self.conv1 = nn.Conv1d(10, 64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(64)
        self.pool = nn.MaxPool1d(2) 
        self.conv2 = nn.Conv1d(64, d_model, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(d_model)
        
        self.pos_encoder = PositionalEncoding(d_model=d_model, max_len=25)
        
        encoder_layers = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=num_heads, 
            dim_feedforward=d_model * 2, 
            dropout=dropout_rate, 
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=num_layers)
        
        self.fc1 = nn.Linear(d_model, 64)
        self.dropout = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = F.relu(self.bn2(self.conv2(x)))
        x = x.permute(0, 2, 1) 
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x)
        x = x.mean(dim=1) 
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x).squeeze(1)


# ==========================================
# 3. THE OPTUNA ENGINE
# ==========================================
# Note: This assumes X_train_scaled, y_train, X_val_scaled, y_val are already in memory 
# from running your master pipeline cell earlier!

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def objective(trial):
    # Determine the mutation geometry
    d_model = trial.suggest_categorical("d_model", [64, 128, 256])
    num_heads = trial.suggest_categorical("num_heads", [2, 4, 8])
    num_layers = trial.suggest_int("num_layers", 1, 3)
    dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5)
    
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [64, 128, 256])

    # Initialize model
    model = TunableConvTransformer(d_model, num_heads, num_layers, dropout_rate).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # Prepare Data
    train_loader = DataLoader(FallDataset(X_train_scaled, y_train), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(FallDataset(X_val_scaled, y_val), batch_size=batch_size, shuffle=False)

    # Fast Training Loop (15 epochs to quickly judge the mutation)
    epochs = 15 
    best_val_f1 = 0.0

    for epoch in range(epochs):
        model.train()
        for b_X, b_y in train_loader:
            b_X, b_y = b_X.to(device), b_y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(b_X), b_y)
            loss.backward()
            optimizer.step()

        model.eval()
        all_preds, all_targets = [], []
        with torch.no_grad():
            for v_X, v_y in val_loader:
                v_X, v_y = v_X.to(device), v_y.to(device)
                logits = model(v_X)
                preds = (torch.sigmoid(logits) > 0.5).float()
                all_preds.extend(preds.cpu().numpy())
                all_targets.extend(v_y.cpu().numpy())

        # Metric
        val_f1 = f1_score(all_targets, all_preds, zero_division=0)
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            
        # Pruning mechanism
        trial.report(val_f1, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return best_val_f1

# ==========================================
# 4. EXECUTE SEARCH
# ==========================================
print("=== INITIATING OPTUNA HYPERPARAMETER SEARCH ===")
# Use MedianPruner to kill obviously bad trials early and save time
study = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner())
study.optimize(objective, n_trials=30)

print("\n=== OPTIMIZATION COMPLETE ===")
print("Best Trial:")
trial = study.best_trial
print(f"  Best F1 Score: {trial.value:.4f}")
print("  Winning Hyperparameters: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

[I 2026-04-03 19:19:38,939] A new study created in memory with name: no-name-e852db9c-5051-4b60-baa6-526752ad2831


=== INITIATING OPTUNA HYPERPARAMETER SEARCH ===


[I 2026-04-03 19:22:23,161] Trial 0 finished with value: 0.90732889158086 and parameters: {'d_model': 256, 'num_heads': 4, 'num_layers': 1, 'dropout_rate': 0.3018944490700276, 'lr': 4.033784839573992e-05, 'batch_size': 64}. Best is trial 0 with value: 0.90732889158086.
[I 2026-04-03 19:24:50,522] Trial 1 finished with value: 0.9068923821039904 and parameters: {'d_model': 256, 'num_heads': 2, 'num_layers': 3, 'dropout_rate': 0.3660842038340082, 'lr': 0.00048235104997398887, 'batch_size': 128}. Best is trial 0 with value: 0.90732889158086.
[I 2026-04-03 19:25:50,211] Trial 2 finished with value: 0.9039615846338536 and parameters: {'d_model': 256, 'num_heads': 8, 'num_layers': 1, 'dropout_rate': 0.33843266373441705, 'lr': 0.001310149937319131, 'batch_size': 256}. Best is trial 0 with value: 0.90732889158086.
[I 2026-04-03 19:27:08,578] Trial 3 finished with value: 0.9116945107398569 and parameters: {'d_model': 256, 'num_heads': 2, 'num_layers': 1, 'dropout_rate': 0.38822976104218276, 'lr'


=== OPTIMIZATION COMPLETE ===
Best Trial:
  Best F1 Score: 0.9157
  Winning Hyperparameters: 
    d_model: 128
    num_heads: 8
    num_layers: 2
    dropout_rate: 0.38574145802807036
    lr: 0.0013845572424527619
    batch_size: 256


In [15]:
class TunedConvTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        
        # --- THE WINNING OPTUNA PARAMETERS ---
        self.d_model = 128
        num_heads = 8
        num_layers = 2
        self.dropout_rate = 0.3857
        
        # 10-Channel Input (9 Sensors + 1 SMV)
        self.conv1 = nn.Conv1d(10, 64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(64)
        self.pool = nn.MaxPool1d(2) 
        self.conv2 = nn.Conv1d(64, self.d_model, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(self.d_model)
        
        self.pos_encoder = PositionalEncoding(d_model=self.d_model, max_len=25)
        
        encoder_layers = nn.TransformerEncoderLayer(
            d_model=self.d_model, 
            nhead=num_heads, 
            dim_feedforward=self.d_model * 2, 
            dropout=self.dropout_rate, 
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=num_layers)
        
        self.fc1 = nn.Linear(self.d_model, 64)
        self.dropout = nn.Dropout(self.dropout_rate)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = F.relu(self.bn2(self.conv2(x)))
        x = x.permute(0, 2, 1) 
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x)
        x = x.mean(dim=1) 
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x).squeeze(1)

In [17]:
# Initialize your highly tuned model
# final_model = TunedConvTransformer()
# criterion = nn.BCEWithLogitsLoss()
# optimizer = optim.Adam(final_model.parameters(), lr=0.00138) 

# # 1. Fire up the Trainer Engine
# trainer = ExperimentTrainer(
#     exp_name="Exp_005_Tuned_Conformer",
#     description="Final run using Optuna's winning parameters (d128, h8, L2, drop0.38).",
#     model=final_model,
#     criterion=criterion,
#     optimizer=optimizer
# )

# # Train for a full 40 epochs to let it reach maximum potential
# best_model, metrics = trainer.train(
#     X_train_scaled, y_train, 
#     X_val_scaled, y_val, 
#     epochs=40, 
#     batch_size=256
# )

# 2. Fire up the Tester Engine
tester = ExperimentTester(
    exp_name="Exp_005_Tuned_Conformer", 
    model_architecture=TunedConvTransformer()
)

# Run the ultimate blind test on the data it has never seen
preds, targets = tester.run_blind_test(
    X_test=X_test_scaled, 
    y_test=y_test, 
    test_name="Ultimate Blind Test", 
    extraction_method="Strict Overlap (50%)",
    batch_size=256
)

print("\n=== PROJECT PIPELINE COMPLETE ===")

Loading locked weights from: models/Exp_005_Tuned_Conformer_best.pth

=== INITIATING BLIND TEST: Ultimate Blind Test ===

              precision    recall  f1-score   support

  Normal (0)       0.99      0.99      0.99      9584
    Fall (1)       0.91      0.91      0.91       957

    accuracy                           0.98     10541
   macro avg       0.95      0.95      0.95     10541
weighted avg       0.98      0.98      0.98     10541

Confusion Matrix:
TN: 9493 | FP: 91
FN: 86 | TP: 871

[Success] Test results securely appended to logs/Exp_005_Tuned_Conformer_report.txt

=== PROJECT PIPELINE COMPLETE ===


## Mamba: State Space Model for Sequence Modeling

### What is Mamba?
Mamba (Gu & Dao, 2023) is a sequence model built on **Selective State Space Models (S4)**,
designed as a direct alternative to the Transformer. Instead of attention, it maintains
a **hidden state** $h_t$ that is continuously updated as it scans the sequence:

$$h_t = \bar{A}h_{t-1} + \bar{B}x_t \qquad y_t = Ch_t + Dx_t$$

Where $\bar{A}$ is the discretized state transition matrix and $\bar{B}$, $C$, $D$
are learnable projection matrices. The key innovation is that $\bar{B}$, $C$, and
the time-step $\Delta$ are **input-dependent** — the model selectively decides what
to remember or forget based on the current token.

---

### Mamba vs Transformer

| Property | Transformer | Mamba |
|----------|-------------|-------|
| Core mechanism | Multi-Head Attention | Selective State Space scan |
| Memory complexity | $\mathcal{O}(L^2)$ | $\mathcal{O}(L)$ |
| Sequence handling | Parallel, global | Sequential, recurrent |
| Long sequences | Expensive | Efficient |
| Positional encoding | Required | Not needed |
| Selective forgetting | No | Yes (input-dependent gates) |

---

### Key Difference in One Line
> A Transformer **looks at everything simultaneously** via attention.
> Mamba **scans the sequence once**, compressing history into a fixed-size
> hidden state — like a smart, selective memory that grows linearly, not quadratically.

---

### Why it Suits Time-Series
- No $\mathcal{O}(L^2)$ bottleneck — handles long sensor sequences efficiently.
- The selective gate naturally suppresses flat, uneventful frames and amplifies
  high-magnitude impact events — without explicit attention weights.

Trying CNN + MAMBA

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ==========================================
# 1. PURE PYTORCH MAMBA (NO C++ COMPILER REQUIRED)
# ==========================================
class MinimalMamba(nn.Module):
    """
    A mathematical recreation of the Mamba architecture using pure PyTorch.
    This bypasses the need for the mamba-ssm C++ kernels.
    """
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.d_inner = int(expand * d_model)
        
        # Input Projections
        self.in_proj = nn.Linear(d_model, self.d_inner * 2)
        self.conv1d = nn.Conv1d(
            in_channels=self.d_inner, out_channels=self.d_inner, 
            kernel_size=d_conv, padding=d_conv - 1, groups=self.d_inner
        )
        
        # Selective Mechanism Parameters
        self.x_proj = nn.Linear(self.d_inner, self.d_state * 2 + 1)
        self.dt_proj = nn.Linear(1, self.d_inner)
        
        # State Matrices (HiPPO Initialization approximation)
        A = torch.arange(1, self.d_state + 1, dtype=torch.float32).repeat(self.d_inner, 1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(self.d_inner))
        
        self.out_proj = nn.Linear(self.d_inner, d_model)

    def forward(self, x):
        b, l, d = x.shape
        
        # 1. Expand input
        x_proj = self.in_proj(x)
        x_ssm, res = x_proj.chunk(2, dim=-1)
        
        # 2. Local Convolution (Time-awareness)
        x_ssm = x_ssm.transpose(1, 2)
        x_ssm = self.conv1d(x_ssm)[:, :, :l] 
        x_ssm = x_ssm.transpose(1, 2)
        x_ssm = F.silu(x_ssm)
        
        # 3. The Selective Mechanism (Dynamically looks at data)
        ssm_params = self.x_proj(x_ssm)
        delta, B, C = torch.split(ssm_params, [1, self.d_state, self.d_state], dim=-1)
        delta = F.softplus(self.dt_proj(delta))
        
        # 4. The State Space Loop (Replaces the Transformer Attention)
        A = -torch.exp(self.A_log.float())
        h = torch.zeros(b, self.d_inner, self.d_state, device=x.device)
        ys = []
        
        # Because sequence length is only 25, this loop runs ultra-fast in pure PyTorch
        for i in range(l):
            x_i = x_ssm[:, i, :]
            dt_i = delta[:, i, :]
            B_i = B[:, i, :]
            C_i = C[:, i, :]
            
            # Mathematical discretization
            dtA = torch.exp(dt_i.unsqueeze(-1) * A)
            dtB = dt_i.unsqueeze(-1) * B_i.unsqueeze(1)
            
            # Update the Hidden State (Mamba's memory)
            h = dtA * h + dtB * x_i.unsqueeze(-1)
            
            # Compute output
            y_i = (h * C_i.unsqueeze(1)).sum(dim=-1) + self.D * x_i
            ys.append(y_i)
            
        y = torch.stack(ys, dim=1)
        y = y * F.silu(res)
        
        return self.out_proj(y)


In [ ]:
# ==========================================
# 2. THE CONV-MAMBA CLASSIFIER
# ==========================================
class ProductionConvMamba(nn.Module):
    def __init__(self):
        super().__init__()
        self.d_model = 128
        self.dropout_rate = 0.38 # We reuse your winning Optuna dropout!
        
        # 1. The CNN Front-End (Exact same as your Conformer)
        self.conv1 = nn.Conv1d(10, 64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(64)
        self.pool = nn.MaxPool1d(2) 
        self.conv2 = nn.Conv1d(64, self.d_model, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(self.d_model)
        
        # 2. The Pure PyTorch Mamba Block
        # Notice: NO Positional Encoding needed!
        self.mamba_block = MinimalMamba(d_model=self.d_model, d_state=16)
        
        # 3. The Classification Head
        self.fc1 = nn.Linear(self.d_model, 64)
        self.dropout = nn.Dropout(self.dropout_rate)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        # Input shape: (Batch, 10, 50)
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = F.relu(self.bn2(self.conv2(x)))
        
        # Switch shape for Mamba: (Batch, SeqLen=25, Channels=128)
        x = x.permute(0, 2, 1) 
        
        # Process through State Space Model
        x = self.mamba_block(x)
        
        # Classify
        x = x.mean(dim=1) 
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x).squeeze(1)

In [ ]:
import torch.optim as optim
# Import  trainer if it's not already in memory
# from utiliti.experiment_trainer import ExperimentTrainer
# from utiliti.experiment_tester import ExperimentTester

print("=== INITIATING MAMBA EXPERIMENT ===")

mamba_model = ProductionConvMamba()
criterion = nn.BCEWithLogitsLoss()

# Mamba models usually prefer a slightly smaller learning rate 
# than Transformers. We will start at 0.0008.
optimizer = optim.Adam(mamba_model.parameters(), lr=0.0008) 

trainer = ExperimentTrainer(
    exp_name="Exp_006_Pure_Mamba",
    description="Testing purely native PyTorch Selective State Space Model vs Conformer.",
    model=mamba_model,
    criterion=criterion,
    optimizer=optimizer
)

# Launch the training!
best_mamba, metrics = trainer.train(
    X_train_scaled, y_train, 
    X_val_scaled, y_val, 
    epochs=30, 
    batch_size=256
)

# Run the Blind Test
tester = ExperimentTester(
    exp_name="Exp_006_Pure_Mamba", 
    model_architecture=ProductionConvMamba()
)

preds, targets = tester.run_blind_test(
    X_test=X_test_scaled, 
    y_test=y_test, 
    test_name="Mamba Blind Test", 
    extraction_method="Strict Overlap (50%)",
    batch_size=256
)

=== INITIATING MAMBA EXPERIMENT ===

=== LAUNCHING EXPERIMENT: Exp_006_Pure_Mamba ===
Device: cuda | Epochs: 30 | Batch: 256
Epoch [01/30] | Train Loss: 0.1513 | Val Loss: 0.0819 | Val F1: 0.8214 | Recall: 0.9235
Epoch [02/30] | Train Loss: 0.0783 | Val Loss: 0.0615 | Val F1: 0.8683 | Recall: 0.9005
Epoch [03/30] | Train Loss: 0.0644 | Val Loss: 0.0815 | Val F1: 0.8340 | Recall: 0.9478
Epoch [04/30] | Train Loss: 0.0584 | Val Loss: 0.0511 | Val F1: 0.8914 | Recall: 0.8968
Epoch [05/30] | Train Loss: 0.0520 | Val Loss: 0.0544 | Val F1: 0.8941 | Recall: 0.9175
Epoch [06/30] | Train Loss: 0.0455 | Val Loss: 0.0654 | Val F1: 0.8771 | Recall: 0.9393
Epoch [07/30] | Train Loss: 0.0434 | Val Loss: 0.0544 | Val F1: 0.8934 | Recall: 0.9405
Epoch [08/30] | Train Loss: 0.0402 | Val Loss: 0.0478 | Val F1: 0.8994 | Recall: 0.9114
Epoch [09/30] | Train Loss: 0.0394 | Val Loss: 0.0557 | Val F1: 0.9004 | Recall: 0.9163
Epoch [10/30] | Train Loss: 0.0361 | Val Loss: 0.0453 | Val F1: 0.9149 | Recall: 0.

using mamba only

In [9]:
class PureMamba(nn.Module):
    def __init__(self):
        super().__init__()
        self.d_model = 128
        self.dropout_rate = 0.38 
        
        # 1. The Projection Layer (Replaces the CNN entirely)
        # Maps the 10 raw sensor channels directly to the 128-dim Mamba brain
        self.input_projection = nn.Linear(10, self.d_model)
        
        # 2. The Pure PyTorch Mamba Block
        # It will now process all 50 frames instead of 25!
        self.mamba_block = MinimalMamba(d_model=self.d_model, d_state=16)
        
        # 3. The Classification Head
        self.fc1 = nn.Linear(self.d_model, 64)
        self.dropout = nn.Dropout(self.dropout_rate)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        # Input shape from WindowExtractor: (Batch, 10 Channels, 50 SeqLen)
        
        # 1. Mamba and Linear layers expect (Batch, SeqLen, Channels)
        # So we permute FIRST.
        x = x.permute(0, 2, 1) # Shape becomes: (Batch, 50, 10)
        
        # 2. Project the raw sensors into high-dimensional space
        x = self.input_projection(x) # Shape becomes: (Batch, 50, 128)
        
        # 3. Process through State Space Model (All 50 frames!)
        x = self.mamba_block(x)
        
        # 4. Classify (Mean pooling across the 50 frames)
        x = x.mean(dim=1) 
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x).squeeze(1)

In [ ]:
import optuna
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import f1_score
import math

# ==========================================
# 1. MISSING DEPENDENCIES (Local to this cell)
# ==========================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=50):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :].to(x.device)

class FallDataset(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y
    def __len__(self): return len(self.y)
    def __getitem__(self, idx):
        # Transpose transforms (SeqLen, Channels) to (Channels, SeqLen) for 1D CNNs
        return torch.tensor(self.X[idx].T, dtype=torch.float32), torch.tensor(self.y[idx], dtype=torch.float32)


# ==========================================
# 2. THE TUNABLE ARCHITECTURE
# ==========================================
class TunePureMamba(nn.Module):
    def __init__(self, d_model, num_heads, num_layers, dropout_rate):
        super().__init__()
        
        # 10-Channel Input (9 Sensors + 1 SMV)
        self.conv1 = nn.Conv1d(10, 64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(64)
        self.pool = nn.MaxPool1d(2) 
        self.conv2 = nn.Conv1d(64, d_model, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(d_model)
        
        self.pos_encoder = PositionalEncoding(d_model=d_model, max_len=25)
        
        encoder_layers = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=num_heads, 
            dim_feedforward=d_model * 2, 
            dropout=dropout_rate, 
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=num_layers)
        
        self.fc1 = nn.Linear(d_model, 64)
        self.dropout = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = F.relu(self.bn2(self.conv2(x)))
        x = x.permute(0, 2, 1) 
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x)
        x = x.mean(dim=1) 
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x).squeeze(1)


# ==========================================
# 3. THE OPTUNA ENGINE
# ==========================================
# Note: This assumes X_train_scaled, y_train, X_val_scaled, y_val are already in memory 
# from running your master pipeline cell earlier!

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def objective(trial):
    # Determine the mutation geometry
    d_model = trial.suggest_categorical("d_model", [64, 128, 256])
    num_heads = trial.suggest_categorical("num_heads", [2, 4, 8])
    num_layers = trial.suggest_int("num_layers", 1, 3)
    dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5)
    
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [64, 128, 256])

    # Initialize model
    model = TunePureMamba(d_model, num_heads, num_layers, dropout_rate).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # Prepare Data
    train_loader = DataLoader(FallDataset(X_train_scaled, y_train), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(FallDataset(X_val_scaled, y_val), batch_size=batch_size, shuffle=False)

    # Fast Training Loop (15 epochs to quickly judge the mutation)
    epochs = 15 
    best_val_f1 = 0.0

    for epoch in range(epochs):
        model.train()
        for b_X, b_y in train_loader:
            b_X, b_y = b_X.to(device), b_y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(b_X), b_y)
            loss.backward()
            optimizer.step()

        model.eval()
        all_preds, all_targets = [], []
        with torch.no_grad():
            for v_X, v_y in val_loader:
                v_X, v_y = v_X.to(device), v_y.to(device)
                logits = model(v_X)
                preds = (torch.sigmoid(logits) > 0.5).float()
                all_preds.extend(preds.cpu().numpy())
                all_targets.extend(v_y.cpu().numpy())

        # Metric
        val_f1 = f1_score(all_targets, all_preds, zero_division=0)
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            
        # Pruning mechanism
        trial.report(val_f1, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return best_val_f1

# ==========================================
# 4. EXECUTE SEARCH
# ==========================================
print("=== INITIATING OPTUNA HYPERPARAMETER SEARCH ===")
# Use MedianPruner to kill obviously bad trials early and save time
study = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner())
study.optimize(objective, n_trials=30)

print("\n=== OPTIMIZATION COMPLETE ===")
print("Best Trial:")
trial = study.best_trial
print(f"  Best F1 Score: {trial.value:.4f}")
print("  Winning Hyperparameters: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

tuned mamba

In [13]:
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F

# 1. THE FINAL HARDCODED ARCHITECTURE
class PureMamba(nn.Module):
    def __init__(self):
        super().__init__()
        # Optuna's Winning Specs
        self.d_model = 64
        self.dropout_rate = 0.3335
        
        # Maps 10 raw channels -> 64-dimensional Mamba space
        self.input_projection = nn.Linear(10, self.d_model)
        
        # The Pure PyTorch Mamba Block
        self.mamba_block = MinimalMamba(d_model=self.d_model, d_state=16)
        
        # Classification Head
        self.fc1 = nn.Linear(self.d_model, 64)
        self.dropout = nn.Dropout(self.dropout_rate)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        # Permute for Linear Layer: (Batch, SeqLen=50, Channels=10)
        x = x.permute(0, 2, 1) 
        
        # Project and process
        x = self.input_projection(x) 
        x = self.mamba_block(x)
        
        # Classify
        x = x.mean(dim=1) 
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x).squeeze(1)


In [ ]:


# 2. THE FINAL EXPERIMENT ENGINE
print("=== LAUNCHING ULTIMATE MAMBA PRODUCTION RUN ===")

final_mamba = PureMamba()
criterion = nn.BCEWithLogitsLoss()
# Optuna's exact winning learning rate
optimizer = optim.Adam(final_mamba.parameters(), lr=0.000483) 

trainer = ExperimentTrainer(
    exp_name="Exp_007_Pure_Mamba",
    description="Final run using Optuna parameters (d_model=64, dropout=0.33, No CNN).",
    model=final_mamba,
    criterion=criterion,
    optimizer=optimizer
)

# Train for 40 epochs to let it settle
best_model, metrics = trainer.train(
    X_train_scaled, y_train, 
    X_val_scaled, y_val, 
    epochs=40, 
    batch_size=128 # Optuna's winning batch size
)

# 3. THE BLIND TEST
tester = ExperimentTester(
    exp_name="Exp_007_Pure_Mamba", 
    model_architecture=PureMamba()
)

preds, targets = tester.run_blind_test(
    X_test=X_test_scaled, 
    y_test=y_test, 
    test_name="Pure Mamba Blind Test", 
    extraction_method="Strict Overlap (50%)",
    batch_size=128
)

=== LAUNCHING ULTIMATE MAMBA PRODUCTION RUN ===

=== LAUNCHING EXPERIMENT: Exp_007_Pure_Mamba ===
Device: cuda | Epochs: 40 | Batch: 128
Epoch [01/40] | Train Loss: 0.1762 | Val Loss: 0.0707 | Val F1: 0.8351 | Recall: 0.8786
Epoch [02/40] | Train Loss: 0.0892 | Val Loss: 0.0713 | Val F1: 0.8383 | Recall: 0.8968
Epoch [03/40] | Train Loss: 0.0794 | Val Loss: 0.0725 | Val F1: 0.8575 | Recall: 0.9053
Epoch [04/40] | Train Loss: 0.0729 | Val Loss: 0.0697 | Val F1: 0.8628 | Recall: 0.9235
Epoch [05/40] | Train Loss: 0.0684 | Val Loss: 0.0654 | Val F1: 0.8642 | Recall: 0.9308
Epoch [06/40] | Train Loss: 0.0653 | Val Loss: 0.0602 | Val F1: 0.8813 | Recall: 0.9187
Epoch [07/40] | Train Loss: 0.0625 | Val Loss: 0.0624 | Val F1: 0.8794 | Recall: 0.9296
Epoch [08/40] | Train Loss: 0.0589 | Val Loss: 0.0583 | Val F1: 0.8905 | Recall: 0.9175
Epoch [09/40] | Train Loss: 0.0581 | Val Loss: 0.0569 | Val F1: 0.8852 | Recall: 0.9175
Epoch [10/40] | Train Loss: 0.0547 | Val Loss: 0.0593 | Val F1: 0.8905 

## Ensemble Implementation: Conformer + Mamba

### Methodology — Weighted Soft-Voting
Rather than training a new model, we loaded the pre-trained `.pth` checkpoints
of both architectures and combined their outputs at inference time across three phases:

1. **Weight Freezing** — gradients frozen (`param.requires_grad = False`),
   both networks set to `eval()` mode to prevent retraining.
2. **Parallel Inference** — the same sensor tensor `[B, 10, 50]` is passed
   through both networks simultaneously, yielding independent raw logits.
3. **Probability Aggregation** — logits converted via Sigmoid, then combined
   using a 60/40 weighted vote:

$$P_{\text{final}} = (0.60 \times P_{\text{conformer}}) + (0.40 \times P_{\text{mamba}})$$

$$\text{Fall Detected if } P_{\text{final}} > 0.50$$

The Conformer received the higher weight because empirical testing showed greater
stability at the $T=50$ context window, allowing it to suppress Mamba's false positives.

---

### Verdict: Rejected for Production

| Metric | Conformer (solo) | Ensemble |
|--------|-----------------|----------|
| False Positives | higher | **69** (lowest) |
| False Negatives | **86** | 122 |

Despite achieving the lowest false positive rate, the ensemble was rejected.
Mamba's lower confidence dragged down the Conformer's predictions, spiking
missed falls from 86 to 122. In medical wearable deployment, a missed fall
is a catastrophic failure — making the **standalone Conformer** the
production-ready architecture.

In [17]:
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader, Dataset # <-- Added Dataset import

# ==========================================
# 1. THE MISSING DATASET CLASS
# ==========================================
class FallDataset(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y
    def __len__(self): return len(self.y)
    def __getitem__(self, idx):
        # Transpose transforms (SeqLen, Channels) to (Channels, SeqLen) for 1D CNNs/Linear layers
        return torch.tensor(self.X[idx].T, dtype=torch.float32), torch.tensor(self.y[idx], dtype=torch.float32)

In [18]:
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader

# 1. THE ENSEMBLE WRAPPER
class DualVotingEnsemble(nn.Module):
    def __init__(self, conformer_model, mamba_model, conformer_weight=0.60):
        super().__init__()
        self.conformer = conformer_model
        self.mamba = mamba_model
        self.c_weight = conformer_weight
        self.m_weight = 1.0 - conformer_weight
        
        # Freeze both models so they don't accidentally update during testing
        for param in self.conformer.parameters(): param.requires_grad = False
        for param in self.mamba.parameters(): param.requires_grad = False
            
        self.conformer.eval()
        self.mamba.eval()

    def forward(self, x):
        # Get raw logits from both models
        logits_c = self.conformer(x)
        logits_m = self.mamba(x)
        
        # Convert logits to probabilities (0.0 to 1.0)
        prob_c = torch.sigmoid(logits_c)
        prob_m = torch.sigmoid(logits_m)
        
        # Apply Weighted Soft Voting
        final_prob = (prob_c * self.c_weight) + (prob_m * self.m_weight)
        
        return final_prob

# 2. LOAD THE SAVED CHAMPIONS
print("=== ASSEMBLING THE ENSEMBLE ===")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Re-initialize the empty architectures (Ensure these classes are in your notebook memory!)
conformer = ConvTransformer().to(device)
mamba = PureMamba().to(device)

# Load their saved weights from the /models folder
conformer.load_state_dict(torch.load("models/Exp_005_Tuned_Conformer_best.pth", map_location=device))
mamba.load_state_dict(torch.load("models/Exp_007_Pure_Mamba_best.pth", map_location=device))

# Create the Ensemble (Giving the Conformer 60% of the voting power since it had fewer False Positives)
ensemble = DualVotingEnsemble(conformer, mamba, conformer_weight=0.60).to(device)

# 3. RUN THE ULTIMATE BLIND TEST
print("=== RUNNING ENSEMBLE INFERENCE ===")
test_loader = DataLoader(FallDataset(X_test_scaled, y_test), batch_size=256, shuffle=False)
all_preds, all_targets = [], []

with torch.no_grad():
    for b_X, b_y in test_loader:
        b_X = b_X.to(device)
        
        # The ensemble returns the final probability directly
        final_probs = ensemble(b_X)
        
        # Threshold at 0.5
        preds = (final_probs > 0.5).float()
        
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(b_y.numpy())

# Calculate and Print the final metrics
print("\n" + classification_report(all_targets, all_preds, target_names=['Normal (0)', 'Fall (1)']))
cm = confusion_matrix(all_targets, all_preds)
print("Ensemble Confusion Matrix:")
print(f"TN: {cm[0][0]} | FP: {cm[0][1]}")
print(f"FN: {cm[1][0]} | TP: {cm[1][1]}")

=== ASSEMBLING THE ENSEMBLE ===


=== RUNNING ENSEMBLE INFERENCE ===

              precision    recall  f1-score   support

  Normal (0)       0.99      0.99      0.99      9584
    Fall (1)       0.92      0.87      0.90       957

    accuracy                           0.98     10541
   macro avg       0.96      0.93      0.94     10541
weighted avg       0.98      0.98      0.98     10541

Ensemble Confusion Matrix:
TN: 9515 | FP: 69
FN: 122 | TP: 835


# END